In [22]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import json
import torch
import pandas as pd

<h3 style="text-align:center"> downloading dataset</h3>

In [23]:
# Load the dataset
dataset = load_dataset("socratesft/SocSci210") 

# Load a mapping file
participant_mapping = hf_hub_download(
    repo_id="socratesft/SocSci210",
    filename="metadata/participant_mapping.json",
    repo_type="dataset"
)

condition_mapping = hf_hub_download(
    repo_id="socratesft/SocSci210",
    filename="metadata/condition_mapping.json",
    repo_type="dataset"
)

task_mapping = hf_hub_download(
    repo_id="socratesft/SocSci210",
    filename="metadata/task_mapping.json",
    repo_type="dataset"
)

print(condition_mapping)

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

/home/jambo/.cache/huggingface/hub/datasets--socratesft--SocSci210/snapshots/048481111a4425ed83dc0eacf15f8431f252b21a/metadata/condition_mapping.json


<h3 style="text-align:center"> checking to see if cuda is installed </h3>

In [24]:
if torch.cuda.is_available():
    print("Cuda")
else:
    print("CPU")

Cuda


<h3 style="text-align:center"> Preprocessing dataset </h3>

In [25]:
print(type(dataset))

<class 'datasets.dataset_dict.DatasetDict'>


In [26]:
dataset = dataset["train"].to_pandas()

In [27]:
dataset.head()

,sample_id,participant,demographic,stimuli,response,condition_num,task_num,prompt,reasoning,study_id
0,0,0,"{'age': 34, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",5,7,0,You are a survey respondent with the following...,"Upon encountering the information about Jaime,...",9nphm
1,1,0,"{'age': 34, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",3,7,1,You are a survey respondent with the following...,"Upon reading the scenario about Jaime, the res...",9nphm
2,2,1,"{'age': 67, 'education': 'Bachelor's degree', ...","You read ""Jaime is 20 years old and was born a...",6,6,0,You are a survey respondent with the following...,"Upon reading the description of Jaime, this re...",9nphm
3,3,1,"{'age': 67, 'education': 'Bachelor's degree', ...","You read ""Jaime is 20 years old and was born a...",5,6,1,You are a survey respondent with the following...,As an individual in their late 60s with a some...,9nphm
4,4,2,"{'age': 79, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",6,4,0,You are a survey respondent with the following...,The respondent is likely to reflect on their o...,9nphm


<p> Breaking down the demographic column into its own dataframe</p>

In [28]:
demographicDataframe = pd.concat([
    dataset[["study_id"]].reset_index(drop=True),
    pd.DataFrame(dataset["demographic"].tolist())
], axis=1)

In [29]:
print(demographicDataframe)

        study_id   age                            education  \
0          9nphm  34.0  Post grad study/professional degree   
1          9nphm  34.0  Post grad study/professional degree   
2          9nphm  67.0                    Bachelor's degree   
3          9nphm  67.0                    Bachelor's degree   
4          9nphm  79.0  Post grad study/professional degree   
...          ...   ...                                  ...   
2901385    ef6my  17.0                                  NaN   
2901386    ef6my  17.0                                  NaN   
2901387    ef6my  17.0                                  NaN   
2901388    ef6my  17.0                                  NaN   
2901389    ef6my  17.0                                  NaN   

                        employment ethnicity  gender  household_size  \
0        Employed as paid employee       NaN  Female             4.0   
1        Employed as paid employee       NaN  Female             4.0   
2                          

<p> to solve the na values problem i decide to fill in values from the previous row </p>

In [30]:
demographicDataframe = demographicDataframe.ffill()

In [31]:
print(demographicDataframe.isna().sum())

study_id                 0
age                      0
education                0
employment               0
ethnicity            16407
gender                   0
household_size           0
housing_ownership        0
housing_type             0
ideology                 0
income                   0
internet_access          0
location                 0
marital_status           0
metro_status             0
party_id                 0
phone_service            0
dtype: int64


noticing ethnicity is full of NA's so will apply backwards fill to it

In [32]:
demographicDataframe["ethnicity"] = demographicDataframe["ethnicity"].bfill()

In [33]:
print(demographicDataframe.isna().sum())

study_id             0
age                  0
education            0
employment           0
ethnicity            0
gender               0
household_size       0
housing_ownership    0
housing_type         0
ideology             0
income               0
internet_access      0
location             0
marital_status       0
metro_status         0
party_id             0
phone_service        0
dtype: int64


In [34]:
demographicDataframe.head()

,study_id,age,education,employment,ethnicity,gender,household_size,housing_ownership,housing_type,ideology,income,internet_access,location,marital_status,metro_status,party_id,phone_service
0,9nphm,34.0,Post grad study/professional degree,Employed as paid employee,White,Female,4.0,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
1,9nphm,34.0,Post grad study/professional degree,Employed as paid employee,White,Female,4.0,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
2,9nphm,67.0,Bachelor's degree,Retired,White,Male,2.0,Owned or being bought by you or someone in you...,A one-family house attached to one or more houses,Somewhat Conservative,150-175K+,Internet Household,Iowa,Married,Metro Area,Don't Lean/Independent/None,Cellphone only
3,9nphm,67.0,Bachelor's degree,Retired,White,Male,2.0,Owned or being bought by you or someone in you...,A one-family house attached to one or more houses,Somewhat Conservative,150-175K+,Internet Household,Iowa,Married,Metro Area,Don't Lean/Independent/None,Cellphone only
4,9nphm,79.0,Post grad study/professional degree,Retired,White,Male,2.0,Owned or being bought by you or someone in you...,A one-family house detached from any other house,Moderate,40-49K,Internet Household,Indiana,Married,Metro Area,Strong Democrat,Cellphone only


merging demographic and dataset together

In [35]:
demographicDataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 2901390 entries, 0 to 2901389
Data columns (total 17 columns):
 #   Column             Dtype  
---  ------             -----  
 0   study_id           str    
 1   age                float64
 2   education          str    
 3   employment         str    
 4   ethnicity          str    
 5   gender             str    
 6   household_size     float64
 7   housing_ownership  str    
 8   housing_type       str    
 9   ideology           str    
 10  income             str    
 11  internet_access    str    
 12  location           str    
 13  marital_status     str    
 14  metro_status       str    
 15  party_id           str    
 16  phone_service      str    
dtypes: float64(2), str(15)
memory usage: 1.1 GB


In [36]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 2901390 entries, 0 to 2901389
Data columns (total 10 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   sample_id      int64 
 1   participant    int64 
 2   demographic    object
 3   stimuli        str   
 4   response       int64 
 5   condition_num  int64 
 6   task_num       int64 
 7   prompt         str   
 8   reasoning      str   
 9   study_id       str   
dtypes: int64(5), object(1), str(4)
memory usage: 7.0+ GB


In [37]:
dataset = pd.merge(dataset.drop_duplicates(subset='study_id'), 
         demographicDataframe.drop_duplicates(subset='study_id'), 
         on='study_id')

In [40]:
dataset.isna().sum()

sample_id            0
participant          0
demographic          0
stimuli              0
response             0
condition_num        0
task_num             0
prompt               0
reasoning            0
study_id             0
age                  0
education            0
employment           0
ethnicity            0
gender               0
household_size       0
housing_ownership    0
housing_type         0
ideology             0
income               0
internet_access      0
location             0
marital_status       0
metro_status         0
party_id             0
phone_service        0
dtype: int64